# 学生端：DNN 用于语音识别（PyTorch）

本 notebook 是学生端课堂训练文件。你只需要按顺序运行代码、观察结果，并在每个 checkpoint 用自己的话回答问题。

注意：本文件不包含标准答案、补讲内容或评分 rubric。你的回答会保存为 `ASR_DNN_checkpoint_records.json`，供教师端生成学习反馈报告。

In [2]:
import sys, json, random
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_records = []
student_name = input('请输入姓名：').strip()
student_id = input('请输入学号：').strip()

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device:', device)

请输入姓名： 王强
请输入学号： 20230612


Python: 3.11.14
PyTorch: 2.1.0+cpu
Device: cpu


In [3]:
def ask_checkpoint(step, title, question, thinking_hint):
    print('\n【Checkpoint】' + title)
    print('思考提示：' + thinking_hint)
    answer = input(question + '\n请用自己的话回答：').strip()
    confidence = input('你对这个回答的把握：高 / 中 / 低：').strip()
    checkpoint_records.append({
        'step': step,
        'title': title,
        'question': question,
        'thinking_hint': thinking_hint,
        'answer': answer,
        'confidence': confidence,
        'timestamp': datetime.now().isoformat(timespec='seconds')
    })
    print('回答已记录。')

## Step 1 任务背景：DNN 声学模型学什么？

真实语音识别系统会从音频中提取 MFCC、FBank 或谱图等声学特征，再由声学模型估计音素、字符、子词或词类别的概率。本节先用小型合成数据模拟“声学特征 -> 数字类别”的过程。

In [4]:
ask_checkpoint(
    'Step 1',
    'DNN 声学模型的输入',
    '为什么本节任务可以理解为“声学特征到类别”的映射，而不是直接让 DNN 处理整段原始 wav？',
    '从语音的短时变化、分帧和特征表示角度思考。'
)


【Checkpoint】DNN 声学模型的输入
思考提示：从语音的短时变化、分帧和特征表示角度思考。


为什么本节任务可以理解为“声学特征到类别”的映射，而不是直接让 DNN 处理整段原始 wav？
请用自己的话回答： 特征向量描述了声道特点，可以捕捉声音类别
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 2 构造简化声学特征数据

In [5]:
labels = np.array(['zero', 'one', 'two', 'three'])
centers = np.array([[0.0, 0.0], [2.5, 0.5], [0.5, 2.5], [2.6, 2.7]], dtype=np.float32)
X_parts, y_parts = [], []
for class_id, center in enumerate(centers):
    X_parts.append(center + np.random.normal(scale=0.55, size=(120, 2)).astype(np.float32))
    y_parts.append(np.full(120, class_id, dtype=np.int64))
X = np.vstack(X_parts).astype(np.float32)
y = np.concatenate(y_parts).astype(np.int64)
print('样本数:', X.shape[0], '特征维度:', X.shape[1], '类别:', labels.tolist())

样本数: 480 特征维度: 2 类别: ['zero', 'one', 'two', 'three']


In [6]:
ask_checkpoint(
    'Step 2',
    '特征维度与类别数',
    '刚才打印出“特征维度=2、类别=4”。这两个数字分别会影响 DNN 的哪一部分设计？',
    '一个数字对应输入样本的表示，另一个数字对应模型要区分的候选结果。'
)


【Checkpoint】特征维度与类别数
思考提示：一个数字对应输入样本的表示，另一个数字对应模型要区分的候选结果。


刚才打印出“特征维度=2、类别=4”。这两个数字分别会影响 DNN 的哪一部分设计？
请用自己的话回答： 输入层和输出层
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 3 划分数据与构造 DataLoader

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)
train_dataset = TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train))
test_tensor = torch.tensor(X_test_scaled).to(device)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
print('训练集:', X_train_scaled.shape, '测试集:', X_test_scaled.shape, 'batch 数:', len(train_loader))

训练集: (360, 2) 测试集: (120, 2) batch 数: 12


In [8]:
ask_checkpoint(
    'Step 3',
    '训练集、测试集与泛化',
    '为什么要留出测试集，并且只在训练集上 fit 标准化器？',
    '从“新语音上的表现”和“测试集信息泄漏”两个角度思考。'
)


【Checkpoint】训练集、测试集与泛化
思考提示：从“新语音上的表现”和“测试集信息泄漏”两个角度思考。


为什么要留出测试集，并且只在训练集上 fit 标准化器？
请用自己的话回答： 为了模型的泛化性
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 4 定义 PyTorch DNN 声学模型

In [9]:
class AcousticDNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, output_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        return self.net(x)

model = AcousticDNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(model)

AcousticDNN(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=4, bias=True)
  )
)


In [10]:
ask_checkpoint(
    'Step 4',
    'DNN 输出层设计',
    '最后一层设成 4 个输出单元，它和本节识别任务之间有什么关系？',
    '看一看 labels 里有几个候选数字。'
)


【Checkpoint】DNN 输出层设计
思考提示：看一看 labels 里有几个候选数字。


最后一层设成 4 个输出单元，它和本节识别任务之间有什么关系？
请用自己的话回答： 本节要进行4分类
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 5 训练循环：logits、loss 与反向传播

In [11]:
ask_checkpoint(
    'Step 5A',
    'CrossEntropyLoss 与 logits',
    '使用 CrossEntropyLoss 时，为什么模型最后一层可以直接输出 logits，而不是先手动转成概率？',
    '从 PyTorch 这个损失函数内部已经做了什么来思考。'
)


【Checkpoint】CrossEntropyLoss 与 logits
思考提示：从 PyTorch 这个损失函数内部已经做了什么来思考。


使用 CrossEntropyLoss 时，为什么模型最后一层可以直接输出 logits，而不是先手动转成概率？
请用自己的话回答： 因为logits出来的就是分类的占比
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


In [12]:
epochs = 80
loss_history = []
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
    avg_loss = total_loss / len(train_dataset)
    loss_history.append(avg_loss)
    if epoch == 1 or epoch % 20 == 0:
        print(f'Epoch {epoch:03d}, loss = {avg_loss:.4f}')

Epoch 001, loss = 1.1353
Epoch 020, loss = 0.0584
Epoch 040, loss = 0.0503
Epoch 060, loss = 0.0432
Epoch 080, loss = 0.0359


In [13]:
ask_checkpoint(
    'Step 5B',
    '梯度清零与参数更新',
    '每个 batch 都执行 zero_grad、backward、step。请说明这三个操作在训练循环中的作用。',
    '分别从清理旧梯度、计算当前梯度、更新参数三个动作解释。'
)


【Checkpoint】梯度清零与参数更新
思考提示：分别从清理旧梯度、计算当前梯度、更新参数三个动作解释。


每个 batch 都执行 zero_grad、backward、step。请说明这三个操作在训练循环中的作用。
请用自己的话回答： 梯度清理，反向传播，权重更新
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 6 评估与简单解码

In [14]:
model.eval()
with torch.no_grad():
    test_logits = model(test_tensor)
    pred_ids = torch.argmax(test_logits, dim=1).cpu().numpy()
acc = accuracy_score(y_test, pred_ids)
print('测试集准确率:', round(acc, 4))
print('\n混淆矩阵：')
print(confusion_matrix(y_test, pred_ids))
print('\n分类报告：')
print(classification_report(y_test, pred_ids, target_names=labels))

测试集准确率: 0.9833

混淆矩阵：
[[30  0  0  0]
 [ 1 29  0  0]
 [ 0  0 29  1]
 [ 0  0  0 30]]

分类报告：
              precision    recall  f1-score   support

        zero       0.97      1.00      0.98        30
         one       1.00      0.97      0.98        30
         two       1.00      0.97      0.98        30
       three       0.97      1.00      0.98        30

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120



In [15]:
ask_checkpoint(
    'Step 6',
    '评估模式与解码',
    '评估时为什么使用 model.eval() 和 torch.no_grad()？argmax 在这里扮演什么角色？',
    '一个问题和评估阶段的模型行为有关，另一个问题和从分数得到类别有关。'
)


【Checkpoint】评估模式与解码
思考提示：一个问题和评估阶段的模型行为有关，另一个问题和从分数得到类别有关。


评估时为什么使用 model.eval() 和 torch.no_grad()？argmax 在这里扮演什么角色？
请用自己的话回答： 模型评估模型，不在计算梯度，分类
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 7 错误分析与课堂迁移

In [16]:
wrong_indices = np.where(pred_ids != y_test)[0]
print('错误样本数量:', len(wrong_indices))
for idx in wrong_indices[:5]:
    print('样本', int(idx), '真实:', labels[y_test[idx]], '预测:', labels[pred_ids[idx]], '特征:', X_test_scaled[idx])

错误样本数量: 2
样本 39 真实: one 预测: zero 特征: [-0.50942564 -1.1387719 ]
样本 44 真实: two 预测: three 特征: [-0.15536429 -0.05765013]


In [17]:
ask_checkpoint(
    'Step 7',
    '从分类评价到 ASR 序列评价',
    '如果从本例的数字分类扩展到完整句子识别，为什么只看普通 accuracy 可能不够？',
    '完整 ASR 输出的是变长文字序列，错误可能是替换、插入或删除。'
)


【Checkpoint】从分类评价到 ASR 序列评价
思考提示：完整 ASR 输出的是变长文字序列，错误可能是替换、插入或删除。


如果从本例的数字分类扩展到完整句子识别，为什么只看普通 accuracy 可能不够？
请用自己的话回答： 还要看错误率等等指标
你对这个回答的把握：高 / 中 / 低： 高


回答已记录。


## Step 8 导出答题记录

运行下面的 cell，将本次 checkpoint 回答保存为 `ASR_DNN_checkpoint_records.json`。

In [18]:
student_record = {
    'student_name': student_name,
    'student_id': student_id,
    'task': 'DNN 用于语音识别',
    'accuracy': float(acc) if 'acc' in globals() else None,
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'checkpoint_records': checkpoint_records
}
with open('ASR_DNN_checkpoint_records.json', 'w', encoding='utf-8') as f:
    json.dump(student_record, f, ensure_ascii=False, indent=2)
print('已保存：ASR_DNN_checkpoint_records.json')
print('checkpoint 数量:', len(checkpoint_records))

已保存：ASR_DNN_checkpoint_records.json
checkpoint 数量: 8
